# FIAP · Fase 6 · Capítulo 1 — YOLOv5 (copo & controle)
**Autor:** Deivisson Gonçalves Lima · **RM565095** · **Grupo 47 (individual)**

Notebook **end-to-end**: coleta do dataset, exportação YOLOv5, treinos (30/60 épocas), validação e inferência.  
Compatível com **CPU ou GPU** no Colab.

## 0) Ambiente e compatibilidades (Colab / Python 3.12)
- Faz *hotfix* de versões para o **FiftyOne** (evita conflitos de `pymongo/mongoengine/pydantic`).  
- No Colab, após instalar o FiftyOne, o runtime **pode reiniciar automaticamente**.

In [ ]:
import sys, os, subprocess

def sh(cmd):
    print(f"$ {cmd}")
    return subprocess.call(cmd, shell=True)

try:
    import fiftyone as fo
    import fiftyone.zoo as foz
    from fiftyone import ViewField as F
    print("FiftyOne já instalado.")
except Exception:
    print("Instalando dependências do FiftyOne...")
    sh("pip uninstall -y fiftyone fiftyone-db fiftyone-brain fiftyone-plugins pymongo mongoengine motor || true")
    sh("pip install -U pymongo==4.6.3 mongoengine==0.24.2 motor==3.4.0")
    sh("pip install -U fiftyone==0.25.0")
    import os; os.kill(os.getpid(), 9)  # força restart no Colab

## 1) Imports e configurações

In [ ]:
import os, glob, random, shutil, yaml
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

random.seed(51)

DATA_DIR = "/content/fase6_data"
os.makedirs(DATA_DIR, exist_ok=True)

CLASSES = ["Coffee cup", "Remote control"]
print("DATA_DIR:", DATA_DIR)
print("CLASSES:", CLASSES)

## 2) Download do Open Images V7 (apenas classes de interesse)

In [ ]:
ds = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=CLASSES,
    max_samples=1200,
    shuffle=True,
)
print("Total de amostras:", len(ds))

schema = ds.get_field_schema()
det_field = None
for fname, ftype in schema.items():
    if "Detections" in str(ftype):
        det_field = fname; break
assert det_field, f"Nenhum campo de Detections no schema: {schema}"
print("Campo de detecção:", det_field)

## 3) Seleção balanceada e *splits* (64/8/8)

In [ ]:
base = ds.filter_labels(det_field, F("label").is_in(CLASSES))

def pick_ids_for_class(view, cls, n, seed=51):
    import random
    random.seed(seed)
    ids = [s.id for s in view.match(F(f"{det_field}.detections").filter(F("label")==cls)).select_fields(det_field)]
    random.shuffle(ids)
    return ids[:n]

ids_by_class = {c: pick_ids_for_class(base, c, 40, 51) for c in CLASSES}
all_ids = sum(ids_by_class.values(), [])
random.shuffle(all_ids)

train_ids, val_ids, test_ids = all_ids[:64], all_ids[64:72], all_ids[72:80]
train_view, val_view, test_view = base.select(train_ids), base.select(val_ids), base.select(test_ids)
print("train/val/test:", len(train_view), len(val_view), len(test_view))

## 4) Exportação YOLOv5 + correção de subpastas

In [ ]:
import shutil, os, glob

shutil.rmtree(DATA_DIR, ignore_errors=True)
os.makedirs(DATA_DIR, exist_ok=True)

def export_split(view, split):
    view.export(
        export_dir=DATA_DIR,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field=det_field,
        classes=CLASSES,
        split=split,
        export_media=True,
        overwrite=True,
    )
    deep = os.path.join(DATA_DIR, split, "images", split)
    if os.path.isdir(deep):
        for fn in os.listdir(deep):
            os.rename(os.path.join(deep, fn), os.path.join(DATA_DIR, split, "images", fn))
        shutil.rmtree(deep, ignore_errors=True)

for s, v in [("train", train_view), ("val", val_view), ("test", test_view)]:
    export_split(v, s)
    imgs = glob.glob(f"{DATA_DIR}/{s}/images/*"); lbls = glob.glob(f"{DATA_DIR}/{s}/labels/*")
    print(s, "imgs:", len(imgs), "| labels:", len(lbls))

## 5) `data.yaml`

In [ ]:
data = {
    "path": DATA_DIR,
    "train": f"{DATA_DIR}/train/images",
    "val":   f"{DATA_DIR}/val/images",
    "test":  f"{DATA_DIR}/test/images",
    "nc": 2,
    "names": ["copo", "controle"],
}
with open(f"{DATA_DIR}/data.yaml", "w") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
print(open(f"{DATA_DIR}/data.yaml").read())

## 6) YOLOv5 — clone e requisitos

In [ ]:
%cd /content
!git clone -q https://github.com/ultralytics/yolov5
%cd /content/yolov5
!pip install -qr requirements.txt --disable-pip-version-check

## 7) Treinos rápidos (30 e 60 épocas)

In [ ]:
DEVICE = "cpu"   # "cpu" ou "0" (GPU)
IMG = 448 if DEVICE == "cpu" else 640
BATCH = 8 if DEVICE == "cpu" else 16

%cd /content/yolov5

!python train.py --img {IMG} --batch {BATCH} --epochs 30   --data {DATA_DIR}/data.yaml --weights yolov5n.pt   --cache ram --workers 2 --device {DEVICE}   --freeze 10 --patience 10   --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs   --name exp_e30_fast_{DEVICE} --exist-ok

!python train.py --img {IMG} --batch {BATCH} --epochs 60   --data {DATA_DIR}/data.yaml --weights yolov5s.pt   --cache ram --workers 2 --device {DEVICE}   --freeze 10 --patience 15   --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs   --name exp_e60_fast_{DEVICE} --exist-ok

## 8) Validação e inferência

In [ ]:
%cd /content/yolov5
BEST30 = "/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e30_fast_cpu/weights/best.pt" if DEVICE=="cpu"          else "/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e30_fast_0/weights/best.pt"
BEST60 = "/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60_fast_cpu/weights/best.pt" if DEVICE=="cpu"          else "/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60_fast_0/weights/best.pt"

!python val.py --weights {BEST30} --data {DATA_DIR}/data.yaml --task val --device {DEVICE}   --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e30_val --exist-ok

!python val.py --weights {BEST60} --data {DATA_DIR}/data.yaml --task val --device {DEVICE}   --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e60_val --exist-ok

!python detect.py --weights {BEST60} --img {IMG} --conf 0.25 --source {DATA_DIR}/val/images --device {DEVICE}   --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name infer_val_e60 --exist-ok

## 9) Conclusões
- **60 épocas** superou **30 épocas** em mAP@0.5 (execuções anteriores: ~0.58 vs ~0.41).  
- A classe **controle** se beneficia mais de mais iterações; **copo** estabiliza cedo.  
- Com **80 imagens** totais, o modelo **generaliza pouco**. Ganhos claros ao crescer a base e aplicar *augmentations*.

**Próximos passos**: aumentar dataset (≥300/classe), *tuning* de LR/scheduler, habilitar `mosaic`/`mixup` com parcimônia e testar `yolov5m/l`/`yolov8`.